In [ ]:
import pandas as pd 

In [ ]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
product_category = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [ ]:
print("customers shape", customers.shape)
print("geolocation shape", geolocation.shape)
print("order_items shape", order_items.shape)
print("order_payments shape", order_payments.shape)
print("order_reviews shape", order_reviews.shape)
print("orders shape", orders.shape)
print("products shape", products.shape)
print("product_category shape", product_category.shape)

In [ ]:
customers.info()

In [ ]:
geolocation.head(3)

In [ ]:
orders.info()

In [ ]:
null_rows = orders[orders.isnull().any(axis=1)]
null_rows

In [ ]:
carrier_date_null = orders["order_delivered_customer_date"].isnull().sum()
carrier_date_null

In [ ]:
customer_date_null = orders["order_delivered_carrier_date"].isnull().sum()
customer_date_null

In [ ]:
estimated_delivery_date_null = orders["order_estimated_delivery_date"].isnull().sum()
estimated_delivery_date_null

In [ ]:
order_status_vs_delivered_customer_date = pd.crosstab(orders["order_status"],orders["order_delivered_customer_date"].isna())
order_status_vs_delivered_customer_date

In [ ]:
order_status_vs_order_delivered_carrier_date  = pd.crosstab(orders["order_status"],orders["order_delivered_carrier_date"].isna())
order_status_vs_order_delivered_carrier_date 

In [ ]:
order_status_vs_order_estimated_delivery_date = pd.crosstab(orders["order_status"],orders["order_estimated_delivery_date"].isna())
order_status_vs_order_estimated_delivery_date

In [ ]:
order_status_vs_order_approved_at  = pd.crosstab(orders["order_status"],orders["order_approved_at"].isna())
order_status_vs_order_approved_at 

In [ ]:
# check whether unavailable orders in the orders table are in the orders_items table 
unaivalable_ids = orders.loc[orders["order_status"] == "unavailable","order_id"]
print(len(unaivalable_ids))

In [ ]:
has_items = unaivalable_ids.isin(order_items["order_id"])
print(has_items.sum())

In [ ]:
six_orders = unaivalable_ids[has_items]
order_items[order_items["order_id"].isin(six_orders)]

In [ ]:
orders[orders["order_id"].isin(six_orders)]

In [ ]:
six_orders.nunique()

In [ ]:
# check if the 6 orders are special from the 603
unavail = orders[orders["order_status"] == "unavailable"]
unavail["order_approved_at"].isna().sum()

In [ ]:
orders.info()

In [ ]:
approved_orders = len(orders[orders['order_status'] == "approved"])
delivered_orders = len(orders[orders['order_status'] == "delivered"])
invoiced_orders = len(orders[orders['order_status'] == "invoiced"])
shipped_orders = len(orders[orders['order_status'] == "shipped"])
processing_orders = len(orders[orders['order_status'] == "processing"])
unavailable_orders = len(orders[orders['order_status'] == "unavailable"])
canceled_orders = len(orders[orders['order_status'] == "canceled"])
created_orders = len(orders[orders['order_status'] == "created"])

In [ ]:
orders["order_status"].unique()

In [ ]:
approved_orders+invoiced_orders+shipped_orders+processing_orders+unavailable_orders+canceled_orders+created_orders

In [ ]:
99441-2963

In [ ]:
delivered_orders

In [ ]:
orders["order_status"].value_counts(dropna=False)

In [ ]:
# orders that can be labeleed are the one ones that were delivered and has customer delivery date 
labelable= orders[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].notna())].copy()
len(labelable)

In [ ]:
labelable["order_status"].unique()

In [ ]:
labelable.head(5)

In [ ]:
labelable.dtypes

In [ ]:
# covert both order_delivered_customer_date and order_estimated_delivery_date 
labelable['order_delivered_customer_date'] = pd.to_datetime(labelable['order_delivered_customer_date'])
labelable['order_estimated_delivery_date'] = pd.to_datetime(labelable['order_estimated_delivery_date'])

In [ ]:
labelable.dtypes

In [ ]:
labelable.head(5)

In [ ]:
print(labelable.columns.tolist())

In [ ]:
labelable = labelable.drop(columns="order_estimated_delivery_date")

In [ ]:
labelable.dtypes

In [ ]:
labelable.head(5)

In [ ]:
# label for late or not late
labelable["is_late"] = (labelable["order_delivered_customer_date"].dt.normalize() > labelable["order_estimated_delivery_date "]).astype(int)

In [ ]:
print(labelable.columns)

In [ ]:
labelable = labelable.rename(columns={"order_estimated_delivery_date ":"order_estimated_delivery_date"})

In [ ]:
print(labelable.columns)

In [ ]:
labelable.head()

In [ ]:
labelable["is_late"].value_counts()

In [ ]:
labelable[labelable["is_late"] == 0]

In [ ]:
labelable["is_late"].isna().sum()